# HW3 Part A - Prompt Engineering Techniques with LangChain

**DATA 266 - Homework 3**

This notebook tries out six different prompt engineering techniques using LangChain. Instead of
screenshotting a chat window, everything here actually calls a model through code - I'm using a
local Ollama model (`llama3.2`) since that's what I had working, per the assignment's note that it
wants to see code calling prompts, not chat screenshots.

The six techniques, with two of my own examples for each:
1. Zero-Shot
2. Few-Shot
3. Chain-of-Thought (CoT)
4. Zero-Shot CoT
5. Meta-Prompting
6. Tree of Thoughts (ToT)


## Step 0 - Personal Parameters

Per the course's standing instructions (Section 0.1), I derive my personal parameters once from
the last four digits of my SJSU ID (SID4) and report them here.


In [1]:
# Step 0: Personal Parameters (Section 0.1 of standing instructions)
import random
import numpy as np

SID4 = 215
SEED = SID4                      # 215
SLICE = SID4 % 1000              # 215
HP_ID = SID4 % 6                 # 5
CLS_A = SID4 % 10                # 5
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10  # 9

random.seed(SEED)
np.random.seed(SEED)

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")

# Note: HP_ID, SLICE, CLS_A, CLS_B do not have a defined mapping for HW3's prompt-engineering
# portion (only HW1 uses HP_ID per the standing instructions), so they are reported for
# reproducibility/traceability but are not otherwise used in this notebook.


SID4  = 215
SEED  = 215
SLICE = 215
HP_ID = 5
CLS_A = 5
CLS_B = 9


## Setup

I'm using [LangChain](https://python.langchain.com/) with a locally hosted **Ollama** model
(`llama3.2`) as the backend. This meant I didn't need a paid API key, and I still got to use all
the normal LangChain building blocks (`PromptTemplate`, `ChatPromptTemplate`,
`FewShotPromptTemplate`, chains, etc.).

Temperature is `0.0` for most techniques so the outputs stay reproducible. Tree-of-Thoughts is the
exception - it uses a higher temperature when generating branches so the different "thoughts"
actually have a chance to differ from each other.


In [2]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, ChatPromptTemplate

MODEL_NAME = "llama3.2"

# Deterministic LLM for most techniques
llm = OllamaLLM(model=MODEL_NAME, temperature=0.0, seed=SEED)

# Slightly stochastic LLM used only for generating diverse ToT branches
llm_branching = OllamaLLM(model=MODEL_NAME, temperature=0.7, seed=SEED)

def run(prompt_text, model=None):
    """Helper: invoke the LLM with a fully-rendered prompt string and print both."""
    model = model or llm
    print("PROMPT:\n" + "-" * 60)
    print(prompt_text)
    print("-" * 60)
    response = model.invoke(prompt_text)
    print("RESPONSE:\n" + "-" * 60)
    print(response)
    print("=" * 60)
    return response


## 1. Zero-Shot Prompting

Here the model just gets the instructions and the question, nothing else - no examples to learn
from. This shows how well it can do purely from the wording of the prompt.

**Example 1: Arithmetic word problem**


In [3]:
zero_shot_prompt_1 = PromptTemplate.from_template(
    "Solve the following math problem and give only the final numeric answer.\n\n"
    "Problem: {problem}\n"
    "Answer:"
)

problem_1 = (
    "A train travels 60 miles in the first hour and 45 miles in the second hour. "
    "What is its average speed in miles per hour over the two hours?"
)

zs1_response = run(zero_shot_prompt_1.format(problem=problem_1))


PROMPT:
------------------------------------------------------------
Solve the following math problem and give only the final numeric answer.

Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?
Answer:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
90


**Example 2: Logical reasoning task (categorical syllogism)**

In [4]:
zero_shot_prompt_2 = PromptTemplate.from_template(
    "Answer the logic question with a single word: Yes, No, or Cannot be determined.\n\n"
    "{statement}\n"
    "Answer:"
)

statement_2 = (
    "All engineers at Acme Corp use Python. Maria uses Python. "
    "Is Maria necessarily an engineer at Acme Corp?"
)

zs2_response = run(zero_shot_prompt_2.format(statement=statement_2))


PROMPT:
------------------------------------------------------------
Answer the logic question with a single word: Yes, No, or Cannot be determined.

All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?
Answer:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
No.


## 2. Few-Shot Prompting

Now I show the model a few solved examples before asking my actual question, using LangChain's
`FewShotPromptTemplate`. The idea is that seeing worked examples nudges it toward the same format
and style.

**Example 1: same arithmetic problem, but now with 3 worked examples first**


In [5]:
example_prompt_math = PromptTemplate.from_template(
    "Problem: {problem}\nAnswer: {answer}"
)

math_examples = [
    {
        "problem": "A car drives 30 miles in 1 hour and 30 miles in the next hour. "
                    "What is its average speed?",
        "answer": "30",
    },
    {
        "problem": "A runner covers 10 miles in the first hour and 20 miles in the second hour. "
                    "What is the average speed?",
        "answer": "15",
    },
    {
        "problem": "A cyclist rides 12 miles in the first hour and 8 miles in the second hour. "
                    "What is the average speed?",
        "answer": "10",
    },
]

few_shot_prompt_1 = FewShotPromptTemplate(
    examples=math_examples,
    example_prompt=example_prompt_math,
    prefix="Solve each problem and give only the final numeric answer (miles per hour).",
    suffix="Problem: {problem}\nAnswer:",
    input_variables=["problem"],
)

fs1_response = run(few_shot_prompt_1.format(problem=problem_1))


PROMPT:
------------------------------------------------------------
Solve each problem and give only the final numeric answer (miles per hour).

Problem: A car drives 30 miles in 1 hour and 30 miles in the next hour. What is its average speed?
Answer: 30

Problem: A runner covers 10 miles in the first hour and 20 miles in the second hour. What is the average speed?
Answer: 15

Problem: A cyclist rides 12 miles in the first hour and 8 miles in the second hour. What is the average speed?
Answer: 10

Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?
Answer:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
To find the average speed, we need to divide the total distance traveled by the total time taken.

Total distance = 60 + 45 = 105 miles
Total time = 1 + 1 = 2 hours

Average speed = Total distance / Total time
= 105 miles / 2 hours
= 52.5 miles per hour


**Example 2: the syllogism question, few-shot with 3 worked examples**

In [6]:
example_prompt_logic = PromptTemplate.from_template(
    "Statement: {statement}\nAnswer: {answer}"
)

logic_examples = [
    {
        "statement": "All cats are mammals. Fluffy is a mammal. Is Fluffy necessarily a cat?",
        "answer": "Cannot be determined",
    },
    {
        "statement": "All squares are rectangles. This shape is a square. Is it necessarily a rectangle?",
        "answer": "Yes",
    },
    {
        "statement": "No fish can fly. A sparrow can fly. Is a sparrow necessarily not a fish?",
        "answer": "Yes",
    },
]

few_shot_prompt_2 = FewShotPromptTemplate(
    examples=logic_examples,
    example_prompt=example_prompt_logic,
    prefix="Answer each logic question with a single phrase: Yes, No, or Cannot be determined.",
    suffix="Statement: {statement}\nAnswer:",
    input_variables=["statement"],
)

fs2_response = run(few_shot_prompt_2.format(statement=statement_2))


PROMPT:
------------------------------------------------------------
Answer each logic question with a single phrase: Yes, No, or Cannot be determined.

Statement: All cats are mammals. Fluffy is a mammal. Is Fluffy necessarily a cat?
Answer: Cannot be determined

Statement: All squares are rectangles. This shape is a square. Is it necessarily a rectangle?
Answer: Yes

Statement: No fish can fly. A sparrow can fly. Is a sparrow necessarily not a fish?
Answer: Yes

Statement: All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?
Answer:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
Cannot be determined


## 3. Chain-of-Thought (CoT) Prompting

This is few-shot, but now the examples also show the reasoning steps, not just the final answer -
basically teaching the model to show its work by example.

**Example 1: same math problem, with worked-out reasoning in the examples**


In [7]:
cot_example_prompt = PromptTemplate.from_template(
    "Problem: {problem}\nReasoning: {reasoning}\nAnswer: {answer}"
)

cot_math_examples = [
    {
        "problem": "A car drives 30 miles in 1 hour and 30 miles in the next hour. "
                    "What is its average speed?",
        "reasoning": "Total distance = 30 + 30 = 60 miles. Total time = 1 + 1 = 2 hours. "
                     "Average speed = 60 / 2 = 30 mph.",
        "answer": "30",
    },
    {
        "problem": "A runner covers 10 miles in the first hour and 20 miles in the second hour. "
                    "What is the average speed?",
        "reasoning": "Total distance = 10 + 20 = 30 miles. Total time = 1 + 1 = 2 hours. "
                     "Average speed = 30 / 2 = 15 mph.",
        "answer": "15",
    },
]

cot_prompt_1 = FewShotPromptTemplate(
    examples=cot_math_examples,
    example_prompt=cot_example_prompt,
    prefix="Solve each problem step by step, then give the final numeric answer.",
    suffix="Problem: {problem}\nReasoning:",
    input_variables=["problem"],
)

cot1_response = run(cot_prompt_1.format(problem=problem_1))


PROMPT:
------------------------------------------------------------
Solve each problem step by step, then give the final numeric answer.

Problem: A car drives 30 miles in 1 hour and 30 miles in the next hour. What is its average speed?
Reasoning: Total distance = 30 + 30 = 60 miles. Total time = 1 + 1 = 2 hours. Average speed = 60 / 2 = 30 mph.
Answer: 30

Problem: A runner covers 10 miles in the first hour and 20 miles in the second hour. What is the average speed?
Reasoning: Total distance = 10 + 20 = 30 miles. Total time = 1 + 1 = 2 hours. Average speed = 30 / 2 = 15 mph.
Answer: 15

Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?
Reasoning:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
I'll solve the problem step by step.

Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?

Step 1: Calculate the total distance traveled by the train.
Total distance = 60 + 45 = 105 miles

Step 2: Calculate the total time taken by the train to travel the total distance.
Total time = 1 + 1 = 2 hours

Step 3: Calculate the average speed of the train.
Average speed = Total distance / Total time
= 105 miles / 2 hours
= 52.5 mph

Answer: 52.5


**Example 2: same syllogism, with worked-out reasoning in the examples**

In [8]:
cot_logic_examples = [
    {
        "problem": "All cats are mammals. Fluffy is a mammal. Is Fluffy necessarily a cat?",
        "reasoning": "The rule only tells us cats are a subset of mammals, not that all mammals "
                     "are cats. Fluffy being a mammal does not place Fluffy specifically in the "
                     "cat subset.",
        "answer": "Cannot be determined",
    },
    {
        "problem": "All squares are rectangles. This shape is a square. Is it necessarily a rectangle?",
        "reasoning": "Since every square is defined as a rectangle, and this shape is a square, "
                     "it must fall inside the rectangle category.",
        "answer": "Yes",
    },
]

cot_prompt_2 = FewShotPromptTemplate(
    examples=cot_logic_examples,
    example_prompt=cot_example_prompt,
    prefix="Reason step by step about each logic question, then answer Yes, No, or Cannot be determined.",
    suffix="Problem: {problem}\nReasoning:",
    input_variables=["problem"],
)

cot2_response = run(cot_prompt_2.format(problem=statement_2))


PROMPT:
------------------------------------------------------------
Reason step by step about each logic question, then answer Yes, No, or Cannot be determined.

Problem: All cats are mammals. Fluffy is a mammal. Is Fluffy necessarily a cat?
Reasoning: The rule only tells us cats are a subset of mammals, not that all mammals are cats. Fluffy being a mammal does not place Fluffy specifically in the cat subset.
Answer: Cannot be determined

Problem: All squares are rectangles. This shape is a square. Is it necessarily a rectangle?
Reasoning: Since every square is defined as a rectangle, and this shape is a square, it must fall inside the rectangle category.
Answer: Yes

Problem: All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?
Reasoning:
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
I'll provide the reasoning and answer for the third problem.

Problem: All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?

Reasoning:

1. The rule states that all engineers at Acme Corp use Python. This means that if someone uses Python, they must be an engineer at Acme Corp.
2. Maria uses Python, which satisfies the condition of using Python.
3. However, the rule does not provide any information about Maria's affiliation with Acme Corp. It only specifies that engineers at Acme Corp use Python.

Conclusion: Since Maria uses Python, but we don't know if she is affiliated with Acme Corp, we cannot determine if Maria is necessarily an engineer at Acme Corp.

Answer: Cannot be determined


## 4. Zero-Shot Chain-of-Thought

This one's a neat trick - no examples at all, you just tack on the phrase **"Let's think step by
step"** to the end of a plain question (this comes from Kojima et al., 2022) and see if that alone
gets the model to reason more carefully.

**Example 1: the same math problem**


In [9]:
zscot_prompt_1 = PromptTemplate.from_template(
    "Problem: {problem}\n"
    "Let's think step by step."
)

zscot1_response = run(zscot_prompt_1.format(problem=problem_1))


PROMPT:
------------------------------------------------------------
Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?
Let's think step by step.
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
To find the average speed of the train, we need to calculate the total distance traveled and divide it by the total time taken.

Step 1: Calculate the total distance traveled
The train traveled 60 miles in the first hour and 45 miles in the second hour. To find the total distance, we add these two distances together:

Total distance = 60 miles + 45 miles = 105 miles

Step 2: Calculate the total time taken
The train traveled for 2 hours.

Step 3: Calculate the average speed
To find the average speed, we divide the total distance by the total time:

Average speed = Total distance / Total time
= 105 miles / 2 hours
= 52.5 miles per hour

Therefore, the average speed of the train is 52.5 miles per hour.


**Example 2: the same syllogism**

In [10]:
zscot_prompt_2 = PromptTemplate.from_template(
    "{statement}\n"
    "Let's think step by step."
)

zscot2_response = run(zscot_prompt_2.format(statement=statement_2))


PROMPT:
------------------------------------------------------------
All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?
Let's think step by step.
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
Let's break down the information step by step:

1. All engineers at Acme Corp use Python.
2. Maria uses Python.

From statement 1, we know that all engineers at Acme Corp use Python. This means that if someone uses Python, they must be an engineer at Acme Corp.

From statement 2, we know that Maria uses Python.

Since Maria uses Python, and we know from statement 1 that all engineers use Python, we can conclude that Maria must be an engineer at Acme Corp.

In other words, the fact that Maria uses Python is sufficient to conclude that she is an engineer at Acme Corp, given the additional information that all engineers at Acme Corp use Python.


## 5. Meta-Prompting

Meta-prompting is about asking the model to first think about *how* it should approach the
problem - what kind of problem is this, what strategy/formula fits - before actually solving it.
So it's a prompt about the approach, rather than showing worked examples.

**Example 1: math problem - model figures out the right approach first**


In [11]:
meta_prompt_1 = PromptTemplate.from_template(
    "You are a meta-reasoning assistant. For the problem below, first (1) classify what type of "
    "problem this is and which mathematical concept/formula applies, then (2) state the general "
    "strategy for solving that class of problem, and finally (3) apply the strategy to compute the "
    "answer.\n\n"
    "Problem: {problem}\n\n"
    "Format your response as:\n"
    "Problem type: ...\n"
    "General strategy: ...\n"
    "Applied solution: ...\n"
    "Final answer: ..."
)

meta1_response = run(meta_prompt_1.format(problem=problem_1))


PROMPT:
------------------------------------------------------------
You are a meta-reasoning assistant. For the problem below, first (1) classify what type of problem this is and which mathematical concept/formula applies, then (2) state the general strategy for solving that class of problem, and finally (3) apply the strategy to compute the answer.

Problem: A train travels 60 miles in the first hour and 45 miles in the second hour. What is its average speed in miles per hour over the two hours?

Format your response as:
Problem type: ...
General strategy: ...
Applied solution: ...
Final answer: ...
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
Problem type: This problem is a classic example of a "two-part" or "two-stage" problem, which involves finding an average value over a given time period. Specifically, it falls under the category of problems that require the application of the formula for average speed, which is:

Average Speed = Total Distance / Total Time

Mathematical concept/formula: Average Speed = Total Distance / Total Time

General strategy: To solve this type of problem, follow these steps:

1. Calculate the total distance traveled by the train over the two hours.
2. Calculate the total time taken by the train to travel the total distance.
3. Divide the total distance by the total time to find the average speed.

Applied solution:

1. Calculate the total distance traveled by the train:
Total Distance = Distance traveled in the first hour + Distance traveled in the second hour
= 60 miles + 45 miles
= 105 miles
2. Calculate the total time tak

**Example 2: syllogism - model first names the logical structure/fallacy involved**

In [12]:
meta_prompt_2 = PromptTemplate.from_template(
    "You are a meta-reasoning assistant. For the statement below, first (1) identify the logical "
    "form involved (e.g., valid syllogism, affirming the consequent, converse error, etc.), then "
    "(2) explain the general rule that governs whether that form is valid, and finally (3) apply "
    "that rule to answer the question.\n\n"
    "{statement}\n\n"
    "Format your response as:\n"
    "Logical form: ...\n"
    "Governing rule: ...\n"
    "Applied answer: ..."
)

meta2_response = run(meta_prompt_2.format(statement=statement_2))


PROMPT:
------------------------------------------------------------
You are a meta-reasoning assistant. For the statement below, first (1) identify the logical form involved (e.g., valid syllogism, affirming the consequent, converse error, etc.), then (2) explain the general rule that governs whether that form is valid, and finally (3) apply that rule to answer the question.

All engineers at Acme Corp use Python. Maria uses Python. Is Maria necessarily an engineer at Acme Corp?

Format your response as:
Logical form: ...
Governing rule: ...
Applied answer: ...
------------------------------------------------------------


RESPONSE:
------------------------------------------------------------
Logical form: 

This is an example of an Affirming the Consequent error, which is a type of fallacy. The correct logical form would be:

All engineers at Acme Corp use Python.
Maria uses Python.
∴ Maria is an engineer at Acme Corp.

However, the given statement is:

All engineers at Acme Corp use Python.
Maria uses Python.
∴ Maria is an engineer at Acme Corp.

The error occurs because the conclusion is drawn based on the fact that Maria uses Python, without considering the fact that not all Python users are engineers at Acme Corp.

Governing rule: 

The governing rule for this form is that if the premises are true, the conclusion is valid only if the conclusion follows logically from the premises. In this case, the premises are true, but the conclusion does not follow logically from the premises because it assumes that all Python users are engineers at Acme Corp, which is not necessarily true.

Applied answer: 

No,

## 6. Tree of Thoughts (ToT)

Tree-of-Thoughts (Yao et al., 2023) is about generating several different reasoning paths for the
same problem, then having the model itself judge which one is best (or combine them into a better
answer). I built a simple version of this with plain LangChain calls:

1. **Generate** a few different candidate solutions ("branches"), each nudged toward a different
   angle so they don't just repeat each other.
2. **Evaluate** the branches with a separate call that critiques them.
3. **Pick** whichever one the judge thinks is most likely correct.

**Example 1: math problem, 3 branches**


In [13]:
def tree_of_thoughts(problem_text, n_branches=3, branch_model=None, judge_model=None):
    branch_model = branch_model or llm_branching
    judge_model = judge_model or llm

    # Each branch gets a distinct persona/instruction so branches diverge even when the
    # underlying Ollama backend seeds its RNG deterministically per call.
    personas = [
        "Solve the problem carefully and directly, showing your arithmetic/logic explicitly.",
        "Solve the problem by first restating it in your own words, then working through it "
        "skeptically, actively looking for ways the obvious answer could be wrong.",
        "Solve the problem by considering a concrete counterexample or edge case before "
        "committing to a final answer.",
    ]

    branch_prompt = PromptTemplate.from_template(
        "{persona}\n\n"
        "Problem: {problem}\n"
        "Reasoning and answer:"
    )

    branches = []
    print(f"Generating {n_branches} candidate reasoning branches...")
    for i in range(n_branches):
        persona = personas[i % len(personas)]
        branch_text = branch_model.invoke(branch_prompt.format(persona=persona, problem=problem_text))
        branches.append(branch_text)
        print(f"\n--- Branch {i+1} ---\n{branch_text}")

    evaluation_prompt = PromptTemplate.from_template(
        "Below are {n} candidate solutions (branches) to the same problem, each produced by an "
        "independent reasoning attempt.\n\n"
        "Problem: {problem}\n\n"
        "{branches_block}\n\n"
        "Evaluate the branches for correctness, briefly explain which one is most likely correct "
        "(or synthesize the correct answer if none are fully right), and end your response with a "
        "line formatted exactly as:\nFINAL ANSWER: <answer>"
    )

    branches_block = "\n\n".join(
        f"Branch {i+1}:\n{b}" for i, b in enumerate(branches)
    )

    verdict = judge_model.invoke(
        evaluation_prompt.format(
            n=n_branches, problem=problem_text, branches_block=branches_block
        )
    )
    print("\n--- Judge's evaluation and final answer ---")
    print(verdict)
    return branches, verdict

tot1_branches, tot1_verdict = tree_of_thoughts(problem_1, n_branches=3)


Generating 3 candidate reasoning branches...



--- Branch 1 ---
To find the average speed of the train over the two hours, we need to divide the total distance traveled by the total time taken.

Total distance traveled = Distance in first hour + Distance in second hour
= 60 miles + 45 miles
= 105 miles

Total time taken = Time taken in first hour + Time taken in second hour
= 1 hour + 1 hour
= 2 hours

Average speed = Total distance traveled / Total time taken
= 105 miles / 2 hours
= 52.5 miles per hour

So, the average speed of the train over the two hours is 52.5 miles per hour.



--- Branch 2 ---
Let's restate the problem in my own words:

I need to find the average speed of a train over a 2-hour period, given that it traveled 60 miles in the first hour and 45 miles in the second hour.

Now, let's work through the problem skeptically:

Initially, I might think that the average speed is simply the total distance traveled divided by the total time, which is (60 + 45) miles / 2 hours. However, this approach assumes that the train's speed remains constant over the 2-hour period, which seems unlikely.

Another potential issue with this approach is that the train's speed in the second hour might be affected by factors such as friction, terrain, or other external factors that could have changed its speed from the first hour.

Let's try a different approach. Suppose the train's speed in the first hour is 60 miles per hour, which is a reasonable assumption. If this is the case, then the train's speed in the second hour should be lower, since it's already spent a lot of


--- Branch 3 ---
To find the average speed of the train over the two hours, we need to divide the total distance traveled by the total time taken.

Total distance traveled = 60 miles (in the first hour) + 45 miles (in the second hour) = 105 miles

Total time taken = 2 hours

Average speed = Total distance traveled / Total time taken
= 105 miles / 2 hours
= 52.5 miles per hour

Therefore, the average speed of the train is 52.5 miles per hour over the two hours.



--- Judge's evaluation and final answer ---
All three branches are correct in their approach to finding the average speed of the train over the two hours. However, Branch 2 provides a more nuanced and skeptical approach, acknowledging the potential issues with assuming constant speed and suggesting alternative methods, such as the harmonic mean formula or the arithmetic mean.

Branch 1 and Branch 3 are straightforward and simple, but they do not provide the same level of critical thinking and consideration of potential issues as Branch 2.

Given the context of the problem, Branch 2 is the most likely correct answer. It provides a clear and logical explanation of the problem and the methods used to solve it, while also acknowledging the potential limitations of the assumptions made.

FINAL ANSWER: 52.5


**Example 2: syllogism, 3 branches**

In [14]:
tot2_branches, tot2_verdict = tree_of_thoughts(statement_2, n_branches=3)


Generating 3 candidate reasoning branches...



--- Branch 1 ---
To solve this problem, we can use a technique called "modus tollens". Modus tollens is a form of logical argument that states:

If A implies B, and not B, then not A.

In this case, let's analyze the statements:

* All engineers at Acme Corp use Python.
* Maria uses Python.

From the first statement, we can infer that "if you are an engineer at Acme Corp, then you use Python". This is a conditional statement, which can be represented as:

E → P (E is an engineer at Acme Corp, P is "you use Python")

The second statement says "Maria uses Python". This is a simple statement that asserts the truth of P for Maria.

Now, let's apply modus tollens to this problem. We can rearrange the first statement to get:

~E → P

This states that "if you are not an engineer at Acme Corp, then you use Python".

Now, we know that Maria uses Python (P), and we want to determine if she is necessarily an engineer at Acme Corp. To do this, we can apply modus tollens as follows:

1. E → P
2. ~


--- Branch 2 ---
Let's restate the problem in our own words:

The problem is asking whether Maria's use of Python necessarily means she is an engineer at Acme Corp.

Now, let's approach this problem skeptically, looking for ways the obvious answer could be wrong.

At first glance, it seems like the obvious answer is "yes." If Maria uses Python, and all engineers at Acme Corp use Python, then it's reasonable to conclude that Maria is an engineer at Acme Corp.

However, let's consider some alternative scenarios that could make the obvious answer incorrect:

1. **Maria is a user, not an engineer**: What if Maria is a user or a customer of Acme Corp, but she's not an engineer? In this case, her use of Python doesn't necessarily imply that she's an engineer.
2. **Maria is an engineer at a different company**: What if Maria is an engineer at a different company that also uses Python? Her use of Python wouldn't necessarily mean she's an engineer at Acme Corp.
3. **There are multiple Python u


--- Branch 3 ---
No, Maria is not necessarily an engineer at Acme Corp.

The information provided is that all engineers at Acme Corp use Python, and Maria uses Python. However, it does not necessarily mean that Maria is an engineer at Acme Corp. She could be a colleague, a manager, or someone else who uses Python for a different reason.

For example, Maria could be a software developer who works on a project that uses Python, but she is not an engineer at Acme Corp. Similarly, she could be a professor teaching a course that uses Python, but she is not an engineer at Acme Corp.

To confirm that Maria is indeed an engineer at Acme Corp, additional information would be needed, such as her job title, department, or any other relevant details.



--- Judge's evaluation and final answer ---
After evaluating the three branches, I would say that Branch 2 is the most likely correct solution. Here's why:

Branch 1 uses modus tollens, which is a valid logical argument, but it assumes that the conditional statement "E → P" is true. However, this assumption is not justified by the information provided. We don't know that all engineers at Acme Corp use Python, only that Maria uses Python.

Branch 2, on the other hand, takes a skeptical approach and considers alternative scenarios that could make the obvious answer incorrect. By doing so, it avoids making assumptions that are not supported by the information provided. This approach is more cautious and less prone to errors.

Branch 3 is incorrect because it assumes that Maria is not an engineer at Acme Corp without providing any evidence. The information provided only states that all engineers at Acme Corp use Python, and Maria uses Python, but it does not necessarily mean that Maria is

## Comparing the Outputs

Below is a table of the final answer each technique gave for my two examples (the train speed
problem, and the "Maria uses Python" syllogism), followed by my notes on what I noticed running all
of this.


In [15]:
import pandas as pd

comparison = pd.DataFrame({
    "Technique": [
        "Zero-Shot", "Few-Shot", "Chain-of-Thought", "Zero-Shot CoT",
        "Meta-Prompting", "Tree of Thoughts",
    ],
    "Example 1 (avg speed, correct = 52.5 mph)": [
        zs1_response.strip().splitlines()[-1] if zs1_response.strip() else "",
        fs1_response.strip().splitlines()[-1] if fs1_response.strip() else "",
        cot1_response.strip().splitlines()[-1] if cot1_response.strip() else "",
        zscot1_response.strip().splitlines()[-1] if zscot1_response.strip() else "",
        meta1_response.strip().splitlines()[-1] if meta1_response.strip() else "",
        tot1_verdict.strip().splitlines()[-1] if tot1_verdict.strip() else "",
    ],
    "Example 2 (syllogism, correct = Cannot be determined)": [
        zs2_response.strip().splitlines()[-1] if zs2_response.strip() else "",
        fs2_response.strip().splitlines()[-1] if fs2_response.strip() else "",
        cot2_response.strip().splitlines()[-1] if cot2_response.strip() else "",
        zscot2_response.strip().splitlines()[-1] if zscot2_response.strip() else "",
        meta2_response.strip().splitlines()[-1] if meta2_response.strip() else "",
        tot2_verdict.strip().splitlines()[-1] if tot2_verdict.strip() else "",
    ],
})
comparison


,Technique,"Example 1 (avg speed, correct = 52.5 mph)","Example 2 (syllogism, correct = Cannot be determined)"
0,Zero-Shot,90,No.
1,Few-Shot,= 52.5 miles per hour,Cannot be determined
2,Chain-of-Thought,Answer: 52.5,Answer: Cannot be determined
3,Zero-Shot CoT,"Therefore, the average speed of the train is 5...","In other words, the fact that Maria uses Pytho..."
4,Meta-Prompting,Final answer: The average speed of the train i...,"No, Maria is not necessarily an engineer at Ac..."
5,Tree of Thoughts,FINAL ANSWER: 52.5,FINAL ANSWER: Maria is not necessarily an engi...


### What I found

**Just so we're clear on the right answers first:**
- The train problem: 105 miles total divided by 2 hours = **52.5 mph**.
- The syllogism: knowing "all engineers use Python" and "Maria uses Python" doesn't actually tell
  you Maria is an engineer - that's the classic affirming-the-consequent mistake. The honest answer
  is **"Cannot be determined."** ("No" is also a fair reading of "is she *necessarily* an engineer,"
  but "cannot be determined" is the more precise way to put it.)

**Here's what each technique actually did (real outputs from `llama3.2`, no cherry-picking):**

- **Zero-Shot** flat-out got the math wrong - it just said `90`, with no work shown, so there's no
  way to tell where that number even came from. On the logic question it said `No`, which is
  basically right, even with zero help.
- **Few-Shot** fixed the math immediately. It was only shown three problem-and-answer pairs (no
  reasoning shown), but it started showing its own work anyway and landed on the correct 52.5. On
  the logic question it gave the clean, correct "Cannot be determined." So the few-shot examples
  didn't really teach it new reasoning - they just seemed to put it into a more careful mode.
- **Chain-of-Thought** (few-shot, but the examples include the reasoning too) also nailed both
  questions, and its answers followed the same structure as my examples pretty closely. Out of all
  six techniques, this one gave the most consistent, predictable output.
- **Zero-Shot CoT** ("let's think step by step") got the math right but **blew the logic
  question** - it reasoned itself all the way to "Maria must be an engineer," which is exactly the
  fallacy the question was testing for. It laid out neat numbered steps the whole way there, which
  is what makes this result interesting: just telling the model to think step by step doesn't mean
  the steps will actually be *valid*. It can sound just as confident being wrong as being right.
- **Meta-Prompting** got both right, and it was the only technique that actually named the fallacy
  out loud ("Affirming the Consequent error") before giving its answer. Its explanation got a
  little tangled at one point, but it still landed on "No, not necessarily an engineer" at the end.
- **Tree of Thoughts** (three different branches - one direct, one skeptical, one looking for
  counterexamples - judged by a fourth call) also got both right, and it was the only technique
  where the branches actually disagreed with each other on the logic question before getting
  resolved. One branch used a shaky argument and still landed close to the right answer, while the
  other two came up with real counterexamples (Maria could be a user, a manager, a professor, etc.)
  showing why the implication doesn't hold. The judge picked up on that and went with the stronger
  reasoning instead of just picking whichever branch sounded most confident.

**Big picture:** both mistakes I saw - zero-shot's wrong math and zero-shot-CoT's bad logic -
happened on the two techniques that don't give the model anything to check its work against. Every
technique that added either a worked example, an explicit "name the strategy" step, or multiple
attempts plus a judge, got both questions right. So it seems like *some* form of double-checking
matters more than which specific technique you use - but just asking the model to "think step by
step" on its own isn't a guarantee it'll reason correctly, only that it'll sound like it's
reasoning. Tree of Thoughts held up the best when one path went wrong, but it's also the most
expensive option - it costs 4 model calls per question here instead of 1.
